# MultVAE Hyperparameter Tuning

## 1. Setup

### 1.1. Imports

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import sys
sys.path.insert(0, '..')

import json
import pickle
import time
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data.loader import load_ratings
from src.data.splitter import split_temporal
from src.data.adapter import DataAdapter
from src.models.multvae import MultVAE
from src.evaluation.metrics import compute_ranking_metrics

### 1.2 Load and split data

In [ ]:
ratings = load_ratings()

train_df, val_df, test_df = split_temporal(
    ratings, 
    train_ratio=0.7, 
    validation_ratio=0.15, 
    test_ratio=0.15,
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Convert to IMPLICIT feedback (rating >= 4 = positive interaction)
adapter = DataAdapter()
train_data = adapter.to_implicit(train_df, threshold=4.0)

print(f"Positive interactions in training: {train_data.X_ui.nnz:,}")

## 2. Tuning

### 2.1 Grid search parameters

In [ ]:
param_grid = {
    'intermediate_dim': [100, 200, 400],
    'latent_dim': [50, 70, 100],
    'n_epochs': [50, 100],
    'beta': [0.5, 1.0],
}

# Generate all combinations
param_names = list(param_grid.keys())
param_values = list(param_grid.values())
all_combinations = list(product(*param_values))

print(f"Total combinations to try: {len(all_combinations)}")

### 2.2 Grid search

In [ ]:
results = []

# Only evaluate against positive validation interactions
val_positive = val_df[val_df['rating'] >= 4]
val_users = val_positive['user_id'].unique()

print(f"Evaluating on {len(val_positive):,} positive interactions from {len(val_users):,} users")

for combo in tqdm(all_combinations, desc="Grid Search"):
    params = dict(zip(param_names, combo))
    
    model = MultVAE(
        intermediate_dim=params['intermediate_dim'],
        latent_dim=params['latent_dim'],
        n_epochs=params['n_epochs'],
        beta=params['beta'],
        seed=42,
    )
    
    start_time = time.time()
    model.fit(train_data)
    train_time = time.time() - start_time
    
    # Generate recommendations for ranking eval
    recommendations = model.recommend_for_eval(val_users, k=10)
    val_ranking = compute_ranking_metrics(val_positive, recommendations, top_k=10)
    
    result = {
        **params,
        'val_ndcg': val_ranking['ndcg'],
        'val_recall': val_ranking['recall'],
        'train_time_sec': train_time,
    }
    results.append(result)
    
    print(f"{params} -> NDCG: {val_ranking['ndcg']:.4f} ({train_time:.1f}s)")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('val_ndcg', ascending=False)
results_df.head(10)

In [ ]:
# Best parameters (by NDCG)
best_row = results_df.iloc[0]
best_params = {
    'intermediate_dim': int(best_row['intermediate_dim']),
    'latent_dim': int(best_row['latent_dim']),
    'n_epochs': int(best_row['n_epochs']),
    'beta': float(best_row['beta']),
}

print("Best parameters:")
print(json.dumps(best_params, indent=2))
print(f"\nValidation NDCG@10: {best_row['val_ndcg']:.4f}")
print(f"Validation Recall@10: {best_row['val_recall']:.4f}")

## 3. Retrain best model

### 3.1 Retrain best model on train+val, evaluate on test

In [ ]:
# Combine train and validation for final model
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
train_val_data = adapter.to_implicit(train_val_df, threshold=4.0)

# Train final model
final_model = MultVAE(**best_params, seed=42)
final_model.fit(train_val_data)

# Ranking eval (NDCG, etc.) - only on positive test interactions
test_positive = test_df[test_df['rating'] >= 4]
test_users = test_positive['user_id'].unique()
recommendations = final_model.recommend_for_eval(test_users, k=10)
ranking_metrics = compute_ranking_metrics(test_positive, recommendations, top_k=10)

print(f"Test NDCG@10: {ranking_metrics['ndcg']:.4f}")
print(f"Test MAP@10: {ranking_metrics['map']:.4f}")
print(f"Test Precision@10: {ranking_metrics['precision']:.4f}")
print(f"Test Recall@10: {ranking_metrics['recall']:.4f}")

### 3.2 Save best model and results

In [ ]:
artifacts_dir = Path('../artifacts')
artifacts_dir.mkdir(exist_ok=True)

# Save parameters
params_to_save = {
    'best_params': best_params,
    'val_ndcg': float(best_row['val_ndcg']),
    'test_ranking_metrics': ranking_metrics,
}
with open(artifacts_dir / 'multvae_params.json', 'w') as f:
    json.dump(params_to_save, f, indent=2)
print(f"Parameters saved to {artifacts_dir / 'multvae_params.json'}")

# Save full results
results_df.to_csv(artifacts_dir / 'multvae_grid_search.csv', index=False)
print(f"Grid search results saved to {artifacts_dir / 'multvae_grid_search.csv'}")

### 3.3 Visualize results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# NDCG vs intermediate_dim
ax = axes[0, 0]
sns.boxplot(data=results_df, x='intermediate_dim', y='val_ndcg', ax=ax)
ax.set_title('NDCG@10 vs Intermediate Dim')

# NDCG vs latent_dim
ax = axes[0, 1]
sns.boxplot(data=results_df, x='latent_dim', y='val_ndcg', ax=ax)
ax.set_title('NDCG@10 vs Latent Dim')

# NDCG vs beta
ax = axes[1, 0]
sns.boxplot(data=results_df, x='beta', y='val_ndcg', ax=ax)
ax.set_title('NDCG@10 vs Beta')

# NDCG vs epochs
ax = axes[1, 1]
sns.boxplot(data=results_df, x='n_epochs', y='val_ndcg', ax=ax)
ax.set_title('NDCG@10 vs Epochs')

plt.tight_layout()
plt.savefig(artifacts_dir / 'multvae_tuning_plots.png', dpi=150)
plt.show()